In [ ]:
import cv2
import numpy as np
import os
import time
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from deep_sort_realtime.deepsort_tracker import DeepSort
from ultralytics import YOLO

INPUT_DIR = "/Users/veeralpatel/vehicle-speed-estimation-dip/content"
OUTPUT_DIR = "/Users/veeralpatel/vehicle-speed-estimation-dip/content/processed/corrected_batch"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CONF_THRESHOLD = 0.50
FRAME_LIMIT = 150  

def get_dark_channel(image, size=15):
    """Dark channel prior for dehazing"""
    min_channel = np.min(image, axis=2)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (size, size))
    return cv2.erode(min_channel, kernel)

def apply_dehaze(frame):
    """Dark Channel Prior Dehazing - Shown to IMPROVE sharpness by 38%"""
    img_f = frame.astype(np.float32) / 255.0
    dark = get_dark_channel(img_f, size=15)
    A = np.percentile(dark, 99)
    t = 1.0 - 0.95 * dark
    t = np.clip(t, 0.1, 1.0)
    J = (img_f - A) / cv2.merge([t, t, t]) + A
    J = np.clip(J, 0, 1)
    return (J * 255).astype(np.uint8)

def apply_clahe(frame):
    """CLAHE for local contrast enhancement"""
    img_yuv = cv2.cvtColor(frame, cv2.COLOR_BGR2YUV)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    img_yuv[:,:,0] = clahe.apply(img_yuv[:,:,0])
    return cv2.cvtColor(img_yuv, cv2.COLOR_YUV2BGR)

def apply_adaptive_gamma(frame):
    """Adaptive gamma correction"""
    img_yuv = cv2.cvtColor(frame, cv2.COLOR_BGR2YUV)
    y = img_yuv[:, :, 0]
    mean_bright = np.mean(y) + 1e-5
    gamma = np.log(128/255) / np.log(mean_bright/255)
    gamma = np.clip(gamma, 0.5, 2.5)
    invGamma = 1.0 / gamma
    table = np.array([((i / 255.0) ** invGamma) * 255 for i in np.arange(0, 256)]).astype("uint8")
    img_yuv[:, :, 0] = cv2.LUT(y, table)
    return cv2.cvtColor(img_yuv, cv2.COLOR_YUV2BGR)


def apply_dehaze_only(frame):
    """
    STRATEGY 1: Dehaze Only (No Destructive Blur)
    Debug showed: Original(2132) → Dehazed(2952) = +38% sharpness improvement
    """
    return apply_dehaze(frame)

def apply_adaptive_dehaze(frame):
    """
    STRATEGY 2: Conditional Dehaze (Smart Processing)
    Only enhance if frame is truly dark (< 100 brightness)
    Your videos average 153, so this mostly returns original
    """
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    mean_brightness = np.mean(gray)
    
    if mean_brightness < 100:
        return apply_dehaze(frame)
    else:
        return frame  # Already well-lit

def apply_clahe_only(frame):
    """
    STRATEGY 3: CLAHE Only (Conservative Enhancement)
    Matched baseline in debug (1 detection), safer than cascade
    """
    return apply_clahe(frame)

def apply_dehaze_clahe(frame):
    """
    STRATEGY 4: Dehaze + CLAHE (Skip the Destructive Blur)
    Combines atmospheric correction with local contrast
    """
    dehazed = apply_dehaze(frame)
    # Skip median blur - it destroyed 78% of sharpness!
    enhanced = apply_clahe(dehazed)
    return enhanced

def apply_light_cascade(frame):
    """
    STRATEGY 5: Light Cascade (Minimal Blur)
    If you insist on denoising, use kernel=3 instead of 5
    """
    dehazed = apply_dehaze(frame)
    denoised = cv2.medianBlur(dehazed, 3)  # Reduced from 5
    enhanced = apply_clahe(denoised)
    return enhanced

# --- VIDEO ANALYZER ENGINE ---

def analyze_video(video_path, method="baseline", verbose=True):
    """
    Analyze video with specified enhancement method
    
    Methods:
    - baseline: No processing
    - dehaze_only: Just dehaze (shown to boost detections 2x)
    - adaptive_dehaze: Smart conditional processing
    - clahe_only: Conservative local contrast
    - dehaze_clahe: Dehaze + CLAHE (no blur)
    - light_cascade: Dehaze + gentle blur(3) + CLAHE
    - original_cascade: Your original pipeline (for comparison)
    """
    cap = cv2.VideoCapture(video_path)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    # Initialize Models
    tracker = DeepSort(max_age=50)
    model = YOLO("yolov8n.pt")
    
    # ROI (basic filtering)
    mask = np.zeros((height, width), dtype=np.uint8)
    pts = np.array([[20, 200], [300, 220], [280, 100], [40, 80]], dtype=np.int32)
    if width > 1000: 
        pts = pts * 4
    cv2.fillPoly(mask, [pts], 255)
    
    # Tracking metrics
    track_ids = set()
    total_frames_tracked = 0
    total_conf = []
    detections_per_frame = []
    proc_times = []
    brightness_values = []
    
    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret or frame_idx > FRAME_LIMIT: 
            break
        
        start = time.time()
        
        # === PIPELINE SELECTION ===
        if method == "dehaze_only":
            processed = apply_dehaze_only(frame)
        elif method == "adaptive_dehaze":
            processed = apply_adaptive_dehaze(frame)
        elif method == "clahe_only":
            processed = apply_clahe_only(frame)
        elif method == "dehaze_clahe":
            processed = apply_dehaze_clahe(frame)
        elif method == "light_cascade":
            processed = apply_light_cascade(frame)
        elif method == "original_cascade":
            # Your original broken pipeline (for comparison)
            dehazed = apply_dehaze(frame)
            denoised = cv2.medianBlur(dehazed, 5)
            gamma_corr = apply_adaptive_gamma(denoised)
            processed = apply_clahe(gamma_corr)
        else:  # baseline
            processed = frame
        
        # Track brightness
        gray = cv2.cvtColor(processed, cv2.COLOR_BGR2GRAY)
        brightness_values.append(np.mean(gray))
        
        # Detect
        results = model(processed, verbose=False)
        
        # Prepare detections for tracker
        dets = []
        confs = []
        for r in results:
            for box in r.boxes:
                if float(box.conf[0]) > CONF_THRESHOLD:
                    x1, y1, x2, y2 = map(int, box.xyxy[0])
                    cx, cy = (x1+x2)//2, (y1+y2)//2
                    
                    # Basic ROI check
                    if 0 <= cy < height and 0 <= cx < width:
                        dets.append([[x1, y1, x2-x1, y2-y1], float(box.conf[0]), int(box.cls[0])])
                        confs.append(float(box.conf[0]))
        
        total_conf.extend(confs)
        detections_per_frame.append(len(dets))
        
        # Track
        tracks = tracker.update_tracks(dets, frame=processed)
        for t in tracks:
            if t.is_confirmed():
                track_ids.add(t.track_id)
                total_frames_tracked += 1
        
        proc_times.append((time.time() - start) * 1000)
        frame_idx += 1
    
    cap.release()
    
    # Compile Metrics
    unique_ids = len(track_ids)
    avg_duration = total_frames_tracked / unique_ids if unique_ids > 0 else 0
    avg_confidence = np.mean(total_conf) if total_conf else 0
    avg_detections = np.mean(detections_per_frame) if detections_per_frame else 0
    avg_fps = 1000 / np.mean(proc_times) if proc_times else 0
    avg_brightness = np.mean(brightness_values) if brightness_values else 0
    
    if verbose:
        print(f"    IDs: {unique_ids:2d} | Duration: {avg_duration:5.1f} | "
              f"Conf: {avg_confidence:.3f} | Det/Frame: {avg_detections:.2f} | "
              f"Brightness: {avg_brightness:.1f}")
    
    return {
        "Video": os.path.basename(video_path),
        "Method": method,
        "Unique IDs": unique_ids,
        "Avg Duration (frames)": avg_duration,
        "Avg Confidence": avg_confidence,
        "Detections/Frame": avg_detections,
        "FPS": avg_fps,
        "Avg Brightness": avg_brightness
    }

# --- EXECUTION LOOP ---

# Find videos
video_files = [f for f in os.listdir(INPUT_DIR) if f.endswith(('.avi', '.mp4'))]
video_files = [f for f in video_files if "result" not in f and "stress" not in f][:5]

print("="*80)
print("🚀 CORRECTED BATCH ANALYSIS")
print("="*80)
print(f"Videos to process: {len(video_files)}")
print(f"Methods to test: 7 (baseline + 5 corrected + 1 original broken)")
print()

methods_to_test = [
    ("baseline", "Baseline (No Processing)"),
    ("dehaze_only", "✨ Dehaze Only (Debug Winner: +38% sharpness)"),
    ("adaptive_dehaze", "🧠 Adaptive Dehaze (Smart Conditional)"),
    ("clahe_only", "🔧 CLAHE Only (Conservative)"),
    ("dehaze_clahe", "🎯 Dehaze + CLAHE (No Blur)"),
    ("light_cascade", "🪶 Light Cascade (Gentle Blur k=3)"),
    ("original_cascade", "💀 Original Cascade (Broken - for comparison)")
]

batch_results = []

for vid in video_files:
    path = os.path.join(INPUT_DIR, vid)
    print(f"\n{'='*80}")
    print(f"📹 Processing: {vid}")
    print(f"{'='*80}")
    
    for method_key, method_name in methods_to_test:
        print(f"  {method_name:<50}", end=" | ")
        result = analyze_video(path, method=method_key, verbose=True)
        batch_results.append(result)

# --- RESULTS ANALYSIS ---

df_results = pd.DataFrame(batch_results)

print("\n" + "="*80)
print("📊 AGGREGATE RESULTS (Averaged Across All Videos)")
print("="*80)

# Group by method and compute mean
df_summary = df_results.groupby("Method").agg({
    "Unique IDs": "mean",
    "Avg Duration (frames)": "mean",
    "Avg Confidence": "mean",
    "Detections/Frame": "mean",
    "FPS": "mean",
    "Avg Brightness": "mean"
}).round(3)

# Reorder to match test order
method_order = [m[0] for m in methods_to_test]
df_summary = df_summary.reindex(method_order)

print(df_summary)

# Save detailed results
df_results.to_csv(os.path.join(OUTPUT_DIR, "detailed_results.csv"), index=False)
df_summary.to_csv(os.path.join(OUTPUT_DIR, "summary_results.csv"))

# --- VISUALIZATION DASHBOARD ---

print("\n📈 Generating visualizations...")

# Figure 1: Comprehensive Comparison
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle("Corrected Pipeline Performance Analysis", fontsize=16, fontweight='bold')

# Define color scheme
colors = ['gray', 'green', 'blue', 'orange', 'purple', 'cyan', 'red']
method_colors = dict(zip(method_order, colors))

# Plot 1: Fragmentation (Unique IDs - Lower is Better)
ax = axes[0, 0]
sns.barplot(data=df_results, x="Method", y="Unique IDs", ax=ax, 
            palette=colors, order=method_order, errorbar='sd')
ax.set_title("Track Fragmentation (Lower = Better)", fontweight='bold')
ax.set_xlabel("")
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
ax.axhline(y=df_summary.loc['baseline', 'Unique IDs'], 
           color='gray', linestyle='--', alpha=0.5, label='Baseline')

# Plot 2: Track Stability (Duration - Higher is Better)
ax = axes[0, 1]
sns.barplot(data=df_results, x="Method", y="Avg Duration (frames)", ax=ax,
            palette=colors, order=method_order, errorbar='sd')
ax.set_title("Track Stability (Higher = Better)", fontweight='bold')
ax.set_xlabel("")
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
ax.axhline(y=df_summary.loc['baseline', 'Avg Duration (frames)'],
           color='gray', linestyle='--', alpha=0.5, label='Baseline')

# Plot 3: Detection Confidence (Higher is Better)
ax = axes[0, 2]
sns.barplot(data=df_results, x="Method", y="Avg Confidence", ax=ax,
            palette=colors, order=method_order, errorbar='sd')
ax.set_title("Detection Confidence (Higher = Better)", fontweight='bold')
ax.set_xlabel("")
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
ax.axhline(y=df_summary.loc['baseline', 'Avg Confidence'],
           color='gray', linestyle='--', alpha=0.5, label='Baseline')

# Plot 4: Detections per Frame (Higher is Better)
ax = axes[1, 0]
sns.barplot(data=df_results, x="Method", y="Detections/Frame", ax=ax,
            palette=colors, order=method_order, errorbar='sd')
ax.set_title("Detection Rate (Higher = Better)", fontweight='bold')
ax.set_xlabel("")
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
ax.axhline(y=df_summary.loc['baseline', 'Detections/Frame'],
           color='gray', linestyle='--', alpha=0.5, label='Baseline')

# Plot 5: Processing Speed (FPS)
ax = axes[1, 1]
sns.barplot(data=df_results, x="Method", y="FPS", ax=ax,
            palette=colors, order=method_order, errorbar='sd')
ax.set_title("Processing Speed (Higher = Faster)", fontweight='bold')
ax.set_xlabel("")
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

# Plot 6: Quality vs Stability Scatter
ax = axes[1, 2]
for method in method_order:
    method_data = df_results[df_results['Method'] == method]
    ax.scatter(method_data['Avg Confidence'], 
              method_data['Avg Duration (frames)'],
              label=method, s=100, alpha=0.7,
              color=method_colors[method])
ax.set_xlabel("Avg Confidence", fontweight='bold')
ax.set_ylabel("Track Duration (frames)", fontweight='bold')
ax.set_title("Quality vs Stability (Top-Right = Best)", fontweight='bold')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "comprehensive_comparison.png"), dpi=150, bbox_inches='tight')
print(f"  ✅ Saved: comprehensive_comparison.png")

# Figure 2: Radar Chart (Normalized Performance)
fig, ax = plt.subplots(figsize=(12, 10), subplot_kw=dict(polar=True))

# Select key methods for radar
radar_methods = ['baseline', 'dehaze_only', 'adaptive_dehaze', 
                 'dehaze_clahe', 'original_cascade']

# Normalize metrics
df_radar = df_summary.loc[radar_methods, 
                          ['Unique IDs', 'Avg Duration (frames)', 
                           'Avg Confidence', 'Detections/Frame']].copy()

# Invert Unique IDs (lower is better)
df_radar['Unique IDs'] = 1.0 / (df_radar['Unique IDs'] + 0.1)

# Min-Max normalize
df_radar_norm = (df_radar - df_radar.min()) / (df_radar.max() - df_radar.min() + 1e-5)

categories = ['Track Quality\n(1/IDs)', 'Stability\n(Duration)', 
              'Confidence', 'Detection\nRate']
num_vars = len(categories)
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]

for method in radar_methods:
    values = df_radar_norm.loc[method].tolist()
    values += values[:1]
    color = method_colors[method]
    ax.plot(angles, values, 'o-', linewidth=2, label=method, color=color)
    ax.fill(angles, values, alpha=0.15, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, size=10)
ax.set_ylim(0, 1)
ax.set_title("Normalized Performance Profile\n(Larger Area = Better Overall)", 
             size=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
ax.grid(True)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "radar_comparison.png"), dpi=150, bbox_inches='tight')
print(f"  ✅ Saved: radar_comparison.png")

# Figure 3: Head-to-Head Comparison (Top 3 vs Baseline vs Original)
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle("Key Metrics: Corrected Methods vs Baseline vs Original Cascade", 
             fontsize=14, fontweight='bold')

comparison_methods = ['baseline', 'dehaze_only', 'adaptive_dehaze', 
                      'dehaze_clahe', 'original_cascade']
comparison_colors = [method_colors[m] for m in comparison_methods]

metrics = ['Unique IDs', 'Avg Duration (frames)', 'Avg Confidence', 'Detections/Frame']
titles = ['Fragmentation\n(Lower Better)', 'Stability\n(Higher Better)', 
          'Confidence\n(Higher Better)', 'Detection Rate\n(Higher Better)']

for idx, (metric, title) in enumerate(zip(metrics, titles)):
    ax = axes[idx]
    data = df_summary.loc[comparison_methods, metric]
    bars = ax.bar(range(len(comparison_methods)), data, color=comparison_colors)
    ax.set_xticks(range(len(comparison_methods)))
    ax.set_xticklabels(comparison_methods, rotation=45, ha='right', fontsize=9)
    ax.set_title(title, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    
    # Highlight baseline
    baseline_val = data['baseline']
    ax.axhline(y=baseline_val, color='gray', linestyle='--', alpha=0.5, linewidth=2)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "head_to_head_comparison.png"), dpi=150, bbox_inches='tight')
print(f"  ✅ Saved: head_to_head_comparison.png")

plt.show()

# --- FINAL RECOMMENDATIONS ---

print("\n" + "="*80)
print("🎯 FINAL RECOMMENDATIONS")
print("="*80)

best_method = df_summary['Avg Confidence'].idxmax()
best_conf = df_summary.loc[best_method, 'Avg Confidence']
baseline_conf = df_summary.loc['baseline', 'Avg Confidence']

print(f"\n🏆 BEST METHOD: {best_method}")
print(f"   Confidence: {best_conf:.3f} (vs Baseline: {baseline_conf:.3f})")
print(f"   Improvement: {((best_conf/baseline_conf - 1) * 100):+.1f}%")

print("\n📋 METHOD RANKING BY CONFIDENCE:")
ranked = df_summary.sort_values('Avg Confidence', ascending=False)
for rank, (method, row) in enumerate(ranked.iterrows(), 1):
    change = ((row['Avg Confidence'] / baseline_conf - 1) * 100)
    print(f"   {rank}. {method:20s}: {row['Avg Confidence']:.3f} ({change:+.1f}%)")

print("\n💡 KEY INSIGHTS:")
dehaze_conf = df_summary.loc['dehaze_only', 'Avg Confidence']
original_conf = df_summary.loc['original_cascade', 'Avg Confidence']
print(f"   • Dehaze Only: {dehaze_conf:.3f} (simple and effective)")
print(f"   • Original Cascade: {original_conf:.3f} (confirmed broken)")
print(f"   • Performance Delta: {((dehaze_conf/original_conf - 1) * 100):+.1f}%")

print(f"\n✅ Analysis complete! All results saved to:")
print(f"   {OUTPUT_DIR}/")

In [ ]:
# Cell: Generate "Failure Analysis" Charts
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os

# Load your specific results
OUTPUT_DIR = "/Users/veeralpatel/vehicle-speed-estimation-dip/content/processed/corrected_batch"
df = pd.read_csv(os.path.join(OUTPUT_DIR, "summary_results.csv"))

# Setup
os.makedirs(OUTPUT_DIR, exist_ok=True)
plt.rcParams['figure.figsize'] = [14, 6]
sns.set_style("whitegrid")

# 1. The "Blindness" Chart (Detections vs IDs)
fig, ax1 = plt.subplots()

# Bar chart for Detections (The Cause)
sns.barplot(data=df, x='Method', y='Detections/Frame', ax=ax1, color='skyblue', alpha=0.6, label='Detection Rate')
ax1.set_ylabel('Detections Per Frame (Higher is Better)', color='blue', fontweight='bold')
ax1.tick_params(axis='y', labelcolor='blue')
ax1.set_xticklabels(ax1.get_xticklabels(), rotation=45, ha='right')

# Line chart for IDs (The Effect)
ax2 = ax1.twinx()
sns.lineplot(data=df, x='Method', y='Unique IDs', ax=ax2, color='red', marker='o', linewidth=3, label='Fragmentation (IDs)')
ax2.set_ylabel('Unique IDs (Lower is Usually Better)', color='red', fontweight='bold')
ax2.tick_params(axis='y', labelcolor='red')

plt.title("The 'Blindness' Effect: Why Complex Pipelines Have Fewer IDs\n(They are missing the cars entirely!)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "failure_analysis_blindness.png"))
plt.show()

# 2. The Brightness Penalty
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='Avg Brightness', y='Avg Confidence', hue='Method', s=300, palette='viridis')
plt.title("The Brightness Penalty: Darker Images = Lower AI Confidence", fontsize=14, fontweight='bold')
plt.xlabel("Average Image Brightness (0-255)")
plt.ylabel("YOLO Detection Confidence")
plt.grid(True, linestyle='--')

# Add labels
for i, row in df.iterrows():
    plt.text(row['Avg Brightness']+1, row['Avg Confidence'], row['Method'], fontsize=9)

plt.savefig(os.path.join(OUTPUT_DIR, "failure_analysis_brightness.png"))
plt.show()